# Project 66 — Local Groundedness Checker

**Score how well an LLM answer is grounded in provided evidence — locally.**

| Component | Choice |
|-----------|--------|
| LLM | Ollama (qwen3:8b) |
| Framework | LangChain |
| Interface | Jupyter |

## 1 · What You Will Learn

1. Build a **groundedness scoring pipeline** for RAG answers
2. Decompose answers into **atomic claims** for granular evaluation
3. Compute **groundedness scores** (% claims evidence-supported)
4. Identify **ungrounded portions** needing attention

## 2 · Why This Project Matters

RAG answers should be grounded in retrieved documents. A groundedness checker
measures this systematically.

## 3 · Setup

In [ ]:
# !pip install -q langchain langchain-ollama pandas

## 4 · Imports

In [ ]:
import json
import pandas as pd
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

MODEL = 'qwen3:8b'
llm = ChatOllama(model=MODEL, temperature=0.1)
print(f'LLM ready: {MODEL}')

## 5 · Test Data

In [ ]:
TEST_PAIRS = [
    {'question': 'Benefits of containerization?',
     'evidence': 'Containers share host kernel, start in seconds, use less memory, ensure dev-prod consistency.',
     'answer': 'Containers are lightweight (shared kernel), start instantly, low memory, consistent environments. Also improve security through isolation.'},
    {'question': 'How does PostgreSQL handle transactions?',
     'evidence': 'PostgreSQL supports ACID compliance, uses MVCC for concurrency, and WAL for durability.',
     'answer': 'PostgreSQL uses ACID compliance, MVCC for concurrent access, and WAL for crash recovery.'},
    {'question': 'What is transfer learning?',
     'evidence': 'Transfer learning uses a pre-trained model as starting point for a different task, reducing data needs.',
     'answer': 'Transfer learning adapts pre-trained models to new tasks. Invented by Andrew Ng, requires 10K+ examples.'},
]
print(f'{len(TEST_PAIRS)} test pairs')

## 6 · Groundedness Checker

In [ ]:
def check_groundedness(evidence, answer):
    prompt = ChatPromptTemplate.from_messages([
        ('system', 'Evaluate groundedness. Return JSON: {"claims": [{"text": "...", "grounded": true/false}], "groundedness_score": 0.0-1.0, "ungrounded_claims": [...], "feedback": "..."}'),
        ('human', 'Evidence: {evidence}\nAnswer: {answer}'),
    ])
    raw = (prompt | llm | StrOutputParser()).invoke({'evidence': evidence, 'answer': answer})
    try:
        s, e = raw.find('{'), raw.rfind('}') + 1
        if s >= 0: return json.loads(raw[s:e])
    except: pass
    return {'groundedness_score': 0.5, 'feedback': 'parse error'}

print('Checker ready.')

## 7 · Run Checks

In [ ]:
results = []
for pair in TEST_PAIRS:
    print(f'Checking: {pair["question"][:40]}...')
    result = check_groundedness(pair['evidence'], pair['answer'])
    result['question'] = pair['question']
    results.append(result)
    print(f'  Score: {result.get("groundedness_score", "?")}')
    for uc in result.get('ungrounded_claims', [])[:3]:
        print(f'    Ungrounded: {str(uc)[:80]}')

## 8 · Summary

In [ ]:
rows = [{'question': r.get('question','?')[:35], 'score': r.get('groundedness_score','?'),
         'feedback': str(r.get('feedback',''))[:50]} for r in results]
print(pd.DataFrame(rows).to_string(index=False))

## 9 · Save

In [ ]:
with open('groundedness_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('Saved.')

## 10 · Limitations & Key Takeaways

**Limitations:** LLM-based scoring, binary grounding, no NLI model

**Takeaways:**
- Groundedness scoring ensures RAG answers stay evidence-supported
- Claim decomposition enables granular quality analysis
- Local evaluation with Ollama keeps costs zero